In [1]:
import calendar
import hashlib
import re
import requests
import sys
import time

from bs4 import BeautifulSoup
from copy import deepcopy
from datetime import date, datetime, timedelta
from pathlib import Path

In [2]:
cookies = {
    'ASP.NET_SessionId': 'o2kzgwvq0jgkvma0zrkkn5x2',
}

headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:150.0) Gecko/20100101 Firefox/150.0',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    # 'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Content-Type': 'application/x-www-form-urlencoded',
    'Origin': 'https://egazzete.mahaonline.gov.in',
    'Connection': 'keep-alive',
    'Referer': 'https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx',
    # 'Cookie': 'ASP.NET_SessionId=o2kzgwvq0jgkvma0zrkkn5x2',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
    'Priority': 'u=0, i',
}

search_form_data = {
    'ScriptManager1_HiddenField': ';;AjaxControlToolkit, Version=1.0.10920.32880, Culture=neutral, PublicKeyToken=28f01b0e84b6d53e:en-GB:816bbca1-959d-46fd-928f-6347d6f2c9c3:e2e86ef9:a9a7729d:9ea3f0e2:9e8e87e9:1df13a87:4c9865be:ba594826:507fcf1b:c7a4182e',
    '__EVENTTARGET': '',
    '__EVENTARGUMENT': '',
    '__LASTFOCUS': '',
    '__VIEWSTATE': '/wEPDwULLTExODc0MzY1OTkPFgIeB0dhemV0dGUykg4AAQAAAP////8BAAAAAAAAAAwCAAAATlN5c3RlbS5EYXRhLCBWZXJzaW9uPTQuMC4wLjAsIEN1bHR1cmU9bmV1dHJhbCwgUHVibGljS2V5VG9rZW49Yjc3YTVjNTYxOTM0ZTA4OQUBAAAAFVN5c3RlbS5EYXRhLkRhdGFUYWJsZQMAAAAZRGF0YVRhYmxlLlJlbW90aW5nVmVyc2lvbglYbWxTY2hlbWELWG1sRGlmZkdyYW0DAQEOU3lzdGVtLlZlcnNpb24CAAAACQMAAAAGBAAAAN8IPD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0idXRmLTE2Ij8+DQo8eHM6c2NoZW1hIHhtbG5zPSIiIHhtbG5zOnhzPSJodHRwOi8vd3d3LnczLm9yZy8yMDAxL1hNTFNjaGVtYSIgeG1sbnM6bXNkYXRhPSJ1cm46c2NoZW1hcy1taWNyb3NvZnQtY29tOnhtbC1tc2RhdGEiPg0KICA8eHM6ZWxlbWVudCBuYW1lPSJUYWJsZTEiPg0KICAgIDx4czpjb21wbGV4VHlwZT4NCiAgICAgIDx4czpzZXF1ZW5jZT4NCiAgICAgICAgPHhzOmVsZW1lbnQgbmFtZT0iUmluIiB0eXBlPSJ4czpzdHJpbmciIG1zZGF0YTp0YXJnZXROYW1lc3BhY2U9IiIgbWluT2NjdXJzPSIwIiAvPg0KICAgICAgICA8eHM6ZWxlbWVudCBuYW1lPSJSZWdObyIgdHlwZT0ieHM6c3RyaW5nIiBtc2RhdGE6dGFyZ2V0TmFtZXNwYWNlPSIiIG1pbk9jY3Vycz0iMCIgLz4NCiAgICAgICAgPHhzOmVsZW1lbnQgbmFtZT0iRnJvbURhdGUiIHR5cGU9InhzOnN0cmluZyIgbXNkYXRhOnRhcmdldE5hbWVzcGFjZT0iIiBtaW5PY2N1cnM9IjAiIC8+DQogICAgICAgIDx4czplbGVtZW50IG5hbWU9IlRvRGF0ZSIgdHlwZT0ieHM6c3RyaW5nIiBtc2RhdGE6dGFyZ2V0TmFtZXNwYWNlPSIiIG1pbk9jY3Vycz0iMCIgLz4NCiAgICAgICAgPHhzOmVsZW1lbnQgbmFtZT0iUHJpY2UiIHR5cGU9InhzOmRlY2ltYWwiIG1zZGF0YTp0YXJnZXROYW1lc3BhY2U9IiIgbWluT2NjdXJzPSIwIiAvPg0KICAgICAgICA8eHM6ZWxlbWVudCBuYW1lPSJQdXJjaGFzZVF1YW50aXR5IiB0eXBlPSJ4czppbnQiIG1zZGF0YTp0YXJnZXROYW1lc3BhY2U9IiIgbWluT2NjdXJzPSIwIiAvPg0KICAgICAgPC94czpzZXF1ZW5jZT4NCiAgICA8L3hzOmNvbXBsZXhUeXBlPg0KICA8L3hzOmVsZW1lbnQ+DQogIDx4czplbGVtZW50IG5hbWU9InRtcERhdGFTZXQiIG1zZGF0YTpJc0RhdGFTZXQ9InRydWUiIG1zZGF0YTpNYWluRGF0YVRhYmxlPSJUYWJsZTEiIG1zZGF0YTpVc2VDdXJyZW50TG9jYWxlPSJ0cnVlIj4NCiAgICA8eHM6Y29tcGxleFR5cGU+DQogICAgICA8eHM6Y2hvaWNlIG1pbk9jY3Vycz0iMCIgbWF4T2NjdXJzPSJ1bmJvdW5kZWQiIC8+DQogICAgPC94czpjb21wbGV4VHlwZT4NCiAgPC94czplbGVtZW50Pg0KPC94czpzY2hlbWE+BgUAAACGAzxkaWZmZ3I6ZGlmZmdyYW0geG1sbnM6bXNkYXRhPSJ1cm46c2NoZW1hcy1taWNyb3NvZnQtY29tOnhtbC1tc2RhdGEiIHhtbG5zOmRpZmZncj0idXJuOnNjaGVtYXMtbWljcm9zb2Z0LWNvbTp4bWwtZGlmZmdyYW0tdjEiPg0KICA8dG1wRGF0YVNldD4NCiAgICA8VGFibGUxIGRpZmZncjppZD0iVGFibGUxMSIgbXNkYXRhOnJvd09yZGVyPSIwIj4NCiAgICAgIDxSaW4gLz4NCiAgICAgIDxSZWdObyAvPg0KICAgICAgPEZyb21EYXRlIC8+DQogICAgICA8VG9EYXRlIC8+DQogICAgICA8UHJpY2U+MC4wPC9QcmljZT4NCiAgICAgIDxQdXJjaGFzZVF1YW50aXR5PjA8L1B1cmNoYXNlUXVhbnRpdHk+DQogICAgPC9UYWJsZTE+DQogIDwvdG1wRGF0YVNldD4NCjwvZGlmZmdyOmRpZmZncmFtPgQDAAAADlN5c3RlbS5WZXJzaW9uBAAAAAZfTWFqb3IGX01pbm9yBl9CdWlsZAlfUmV2aXNpb24AAAAACAgICAIAAAAAAAAA//////////8LFgJmD2QWAgIDD2QWAgIJD2QWBAIHD2QWBGYPZBYEAgEPZBYCAgEPEA8WBh4NRGF0YVRleHRGaWVsZAUMRElWSVNJT05OQU1FHg5EYXRhVmFsdWVGaWVsZAUKRElWSVNJT05JRB4LXyFEYXRhQm91bmRnZBAVCBAtLS0tLVNlbGVjdC0tLS0tD0NFTlRSQUwgU0VDVElPTg5LT0tBTiBESVZJU0lPTg1QVU5FIERJVklTSU9ODk5BU0lLIERJVklTSU9OD05BR1BVUiBESVZJU0lPThFBTVJBVkFUSSBESVZJU0lPThNBVVJBTkdBQkFEIERJVklTSU9OFQgQLS0tLS1TZWxlY3QtLS0tLQExATIBMwE0ATUBNgE3FCsDCGdnZ2dnZ2dnZGQCAw9kFgICAQ8QDxYGHwEFC1NFQ1RJT05OQU1FHwIFCVNFQ1RJT05JRB8DZ2QQFRIQLS0tLS1TZWxlY3QtLS0tLQhQYXJ0IC0gMQlQYXJ0IC0xIEwJUGFydCAtMSBBElBhcnQgdHdvIChTYW5raXJuKRVQYXJ0IHR3byAoc3VwcGxpbWVudCkZUGFydCB0d28gKENoYW5nZSBJbiBOYW1lKRBQYXJ0IDQgKE1hcmF0aGkpClBhcnQgNCAtIEEJUGFydCAtNCBCCFBhcnQgNCBDEFBhcnQgNSAoTWFyYXRoaSkRUGFydCA1QSAoRW5nbGlzaCkHUGFydCAtNg5QYXJ0IDcgKEhpbmRpKRBQYXJ0IDggKEVuZ2xpc2gpI01haGFyYXNodHJhIFNoYXNhbiBSYWpwYXRyYSBQYXJ0LUlJJE1haGFyYXNodHJhIFNoYXNhbiBSYWpwYXRyYSBQYXJ0LVhJSRUSEC0tLS0tU2VsZWN0LS0tLS0BMQEyATMBNAE1ATYBNwE4ATkCMTACMTECMTICMTMCMTQCMTUCMTcCMTgUKwMSZ2dnZ2dnZ2dnZ2dnZ2dnZ2dnFgECBmQCAg9kFgICAw9kFgICAQ8PFgIeBFRleHRlZGQCCw88KwARAgEQFgAWABYADBQrAABkGAEFE2N0bDAwJENQSCRHcmlkVmlldzIPZ2QvFxqU1pLdexoF7mOT3Wp7F+YyTqh+OcBZzQuqhkdeqA==',
    '__VIEWSTATEGENERATOR': 'C7385081',
    'ctl00$CPH$ddldivision': '3',
    'ctl00$CPH$ddlSection': '6',
    'ctl00$CPH$txtfromDate': '',
    'ctl00$CPH$txtToDate': '',
    'ctl00$CPH$ddlGazetteType': '0',
    'ctl00$CPH$btnSearch': 'Search',
}
BASE_URL = "https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx"
OUTPUT_DIR = Path("gazette_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:

def extract_hidden_fields(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    hidden = {}
    for el in soup.select("input[type='hidden'][name]"):
        hidden[el.get("name")] = el.get("value", "")
    return hidden


def extract_view_targets(html: str):
    soup = BeautifulSoup(html, "html.parser")
    targets = []

    for a in soup.select('a[id^="CPH_GridView2_LinkButton1_"]'):
        href = (a.get("href") or "").strip()
        m = re.search(r"__doPostBack\('([^']+)',''\)", href)
        if m:
            targets.append(m.group(1))

    return targets


def save_response(prefix: str, resp: requests.Response):
    ct = resp.headers.get("Content-Type", "")
    suffix = ".bin"
    if "application/pdf" in ct.lower():
        suffix = ".pdf"
    elif "html" in ct.lower():
        suffix = ".html"

    out = OUTPUT_DIR / f"{prefix}{suffix}"
    out.write_bytes(resp.content)
    return out


def print_response_summary(label: str, resp: requests.Response):
    print(f"\n--- {label} ---")
    print("URL         :", resp.url)
    print("Status      :", resp.status_code)
    print("Content-Type:", resp.headers.get("Content-Type"))
    print("Location    :", resp.headers.get("Location"))
    print("Length      :", len(resp.content))


def safe_name(value: str, max_len: int = 80) -> str:
    value = (value or "").strip()
    value = re.sub(r"[^\w\-\.]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("._")
    if not value:
        value = "item"
    return value[:max_len]


def target_to_prefix(window_index: int, row_index: int, event_target: str) -> str:
    short_hash = hashlib.sha1(event_target.encode("utf-8")).hexdigest()[:10]
    return f"w{window_index + 1:02d}_{row_index + 1:03d}_{safe_name(event_target, 60)}_{short_hash}"


def parse_yyyy_mm_dd(s: str) -> date:
    return datetime.strptime(s, "%Y-%m-%d").date()


def format_dd_mm_yyyy(d: date) -> str:
    return d.strftime("%d/%m/%Y")


def add_one_year(d: date) -> date:
    target_year = d.year + 1
    target_month = d.month
    target_day = min(d.day, calendar.monthrange(target_year, target_month)[1])
    return date(target_year, target_month, target_day)


def year_chunks(start_date: date, end_date: date):
    if start_date > end_date:
        raise ValueError("start_date must be <= end_date")

    chunks = []
    current_start = start_date

    while current_start <= end_date:
        next_year_same_day = add_one_year(current_start)
        current_end = min(next_year_same_day - timedelta(days=1), end_date)
        chunks.append((current_start, current_end))
        current_start = current_end + timedelta(days=1)

    return chunks


def run_search_and_download_window(
    session: requests.Session,
    base_search_form_data: dict,
    start_date: date,
    end_date: date,
    window_index: int,
    sleep_seconds: float = 1.0,
):
    window_form_data = deepcopy(base_search_form_data)
    window_form_data["ctl00$CPH$txtfromDate"] = format_dd_mm_yyyy(start_date)
    window_form_data["ctl00$CPH$txtToDate"] = format_dd_mm_yyyy(end_date)

    print(f"\n========== WINDOW {window_index + 1} ==========")
    print("From:", window_form_data["ctl00$CPH$txtfromDate"])
    print("To  :", window_form_data["ctl00$CPH$txtToDate"])

    print("\nSubmitting search request...")
    search_resp = session.post(
        BASE_URL,
        data=window_form_data,
        allow_redirects=True,
        timeout=600,
    )
    print_response_summary(f"SEARCH RESPONSE WINDOW {window_index + 1}", search_resp)

    search_prefix = f"window_{window_index + 1:02d}_{start_date.isoformat()}_to_{end_date.isoformat()}_search_response"
    search_file = save_response(search_prefix, search_resp)
    print("Saved       :", search_file)

    search_html = search_resp.text
    hidden_from_results = extract_hidden_fields(search_html)
    view_targets = extract_view_targets(search_html)

    print("\nHidden fields found in search response:")
    for k in sorted(hidden_from_results.keys()):
        if k.startswith("__") or "ScriptManager" in k:
            print(" ", k)

    print(f"\nView targets found: {len(view_targets)}")
    for i, tgt in enumerate(view_targets[:10]):
        print(f"  [{i}] {tgt}")

    downloaded = []
    failed = []

    for row_index, event_target in enumerate(view_targets):
        view_form_data = deepcopy(window_form_data)

        for key in [
            "__VIEWSTATE",
            "__VIEWSTATEGENERATOR",
            "__EVENTVALIDATION",
            "ScriptManager1_HiddenField",
            "__LASTFOCUS",
        ]:
            if key in hidden_from_results:
                view_form_data[key] = hidden_from_results[key]

        view_form_data["__EVENTTARGET"] = event_target
        view_form_data["__EVENTARGUMENT"] = ""
        view_form_data.pop("ctl00$CPH$btnSearch", None)

        print(f"\n[{row_index + 1}/{len(view_targets)}] Submitting View postback for: {event_target}")

        try:
            view_resp = session.post(
                BASE_URL,
                data=view_form_data,
                allow_redirects=False,
                timeout=600,
            )
            print_response_summary(f"VIEW RESPONSE W{window_index + 1} #{row_index + 1}", view_resp)

            prefix = target_to_prefix(window_index, row_index, event_target)
            saved_file = save_response(prefix, view_resp)
            print("Saved       :", saved_file)

            ct = (view_resp.headers.get("Content-Type") or "").lower()
            if "application/pdf" in ct and view_resp.status_code == 200:
                downloaded.append((event_target, saved_file))
            else:
                failed.append((event_target, saved_file, view_resp.status_code, ct, view_resp.headers.get("Location")))

        except requests.RequestException as exc:
            print("ERROR       :", repr(exc))
            failed.append((event_target, None, None, "request_exception", str(exc)))

        time.sleep(sleep_seconds)

    print(f"\n=== WINDOW {window_index + 1} SUMMARY ===")
    print("Range         :", start_date.isoformat(), "to", end_date.isoformat())
    print("Total targets :", len(view_targets))
    print("Downloaded    :", len(downloaded))
    print("Failed/nonpdf :", len(failed))

    return {
        "window_index": window_index,
        "start_date": start_date,
        "end_date": end_date,
        "search_file": search_file,
        "view_targets": view_targets,
        "downloaded": downloaded,
        "failed": failed,
    }

In [4]:
user_start_date = parse_yyyy_mm_dd("2019-08-01")
user_end_date = parse_yyyy_mm_dd("2025-12-31")

In [5]:
if not cookies:
    print("ERROR: cookies dict is empty.")
    sys.exit(1)

if not headers:
    print("ERROR: headers dict is empty.")
    sys.exit(1)

if not search_form_data:
    print("ERROR: search_form_data dict is empty.")
    sys.exit(1)

windows = year_chunks(user_start_date, user_end_date)

print("\nYear windows to be queried:")
for i, (ws, we) in enumerate(windows, start=1):
    print(f"  [{i:02d}] {ws.isoformat()} -> {we.isoformat()}")

s = requests.Session()
s.cookies.update(cookies)
s.headers.update(headers)

all_results = []

for window_index, (window_start, window_end) in enumerate(windows):
    result = run_search_and_download_window(
        session=s,
        base_search_form_data=search_form_data,
        start_date=window_start,
        end_date=window_end,
        window_index=window_index,
        sleep_seconds=1.0,
    )
    all_results.append(result)
    time.sleep(2.0)

print("\n========== OVERALL SUMMARY ==========")
print("Windows processed:", len(all_results))
print("Total PDFs       :", sum(len(r["downloaded"]) for r in all_results))
print("Total failures   :", sum(len(r["failed"]) for r in all_results))


Year windows to be queried:
  [01] 2019-08-01 -> 2020-07-31
  [02] 2020-08-01 -> 2021-07-31
  [03] 2021-08-01 -> 2022-07-31
  [04] 2022-08-01 -> 2023-07-31
  [05] 2023-08-01 -> 2024-07-31
  [06] 2024-08-01 -> 2025-07-31
  [07] 2025-08-01 -> 2025-12-31

========== WINDOW 1 ==========
From: 01/08/2019
To  : 31/07/2020

Submitting search request...

--- SEARCH RESPONSE WINDOW 1 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: text/html; charset=utf-8
Location    : None
Length      : 139543
Saved       : gazette_output/window_01_2019-08-01_to_2020-07-31_search_response.html

Hidden fields found in search response:
  ScriptManager1_HiddenField
  __EVENTARGUMENT
  __EVENTTARGET
  __LASTFOCUS
  __VIEWSTATE
  __VIEWSTATEGENERATOR

View targets found: 53
  [0] ctl00$CPH$GridView2$ctl02$LinkButton1
  [1] ctl00$CPH$GridView2$ctl03$LinkButton1
  [2] ctl00$CPH$GridView2$ctl04$LinkButton1
  [3] ctl00$CPH$GridView2$ctl05$LinkButton1
  [4] 


[21/53] Submitting View postback for: ctl00$CPH$GridView2$ctl22$LinkButton1

--- VIEW RESPONSE W1 #21 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 788998
Saved       : gazette_output/w01_021_ctl00_CPH_GridView2_ctl22_LinkButton1_917bfdf476.pdf

[22/53] Submitting View postback for: ctl00$CPH$GridView2$ctl23$LinkButton1

--- VIEW RESPONSE W1 #22 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 650570
Saved       : gazette_output/w01_022_ctl00_CPH_GridView2_ctl23_LinkButton1_d1a776194f.pdf

[23/53] Submitting View postback for: ctl00$CPH$GridView2$ctl24$LinkButton1

--- VIEW RESPONSE W1 #23 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 851057
Saved       :


[44/53] Submitting View postback for: ctl00$CPH$GridView2$ctl45$LinkButton1

--- VIEW RESPONSE W1 #44 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 638743
Saved       : gazette_output/w01_044_ctl00_CPH_GridView2_ctl45_LinkButton1_9b0c7ba825.pdf

[45/53] Submitting View postback for: ctl00$CPH$GridView2$ctl46$LinkButton1

--- VIEW RESPONSE W1 #45 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 604992
Saved       : gazette_output/w01_045_ctl00_CPH_GridView2_ctl46_LinkButton1_5da90fa914.pdf

[46/53] Submitting View postback for: ctl00$CPH$GridView2$ctl47$LinkButton1

--- VIEW RESPONSE W1 #46 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 708508
Saved       :


[11/52] Submitting View postback for: ctl00$CPH$GridView2$ctl12$LinkButton1

--- VIEW RESPONSE W2 #11 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 367015
Saved       : gazette_output/w02_011_ctl00_CPH_GridView2_ctl12_LinkButton1_5de5e17682.pdf

[12/52] Submitting View postback for: ctl00$CPH$GridView2$ctl13$LinkButton1

--- VIEW RESPONSE W2 #12 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 399573
Saved       : gazette_output/w02_012_ctl00_CPH_GridView2_ctl13_LinkButton1_320aa84bf2.pdf

[13/52] Submitting View postback for: ctl00$CPH$GridView2$ctl14$LinkButton1

--- VIEW RESPONSE W2 #13 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 441934
Saved       :


[34/52] Submitting View postback for: ctl00$CPH$GridView2$ctl35$LinkButton1

--- VIEW RESPONSE W2 #34 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 765625
Saved       : gazette_output/w02_034_ctl00_CPH_GridView2_ctl35_LinkButton1_27d4c533d5.pdf

[35/52] Submitting View postback for: ctl00$CPH$GridView2$ctl36$LinkButton1

--- VIEW RESPONSE W2 #35 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 653537
Saved       : gazette_output/w02_035_ctl00_CPH_GridView2_ctl36_LinkButton1_320ca113f1.pdf

[36/52] Submitting View postback for: ctl00$CPH$GridView2$ctl37$LinkButton1

--- VIEW RESPONSE W2 #36 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 675111
Saved       :

...


[29/52] Submitting View postback for: ctl00$CPH$GridView2$ctl30$LinkButton1

--- VIEW RESPONSE W5 #29 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 760448
Saved       : gazette_output/w05_029_ctl00_CPH_GridView2_ctl30_LinkButton1_055ceb9470.pdf

[30/52] Submitting View postback for: ctl00$CPH$GridView2$ctl31$LinkButton1

--- VIEW RESPONSE W5 #30 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 804614
Saved       : gazette_output/w05_030_ctl00_CPH_GridView2_ctl31_LinkButton1_aeb34ef1c6.pdf

[31/52] Submitting View postback for: ctl00$CPH$GridView2$ctl32$LinkButton1

--- VIEW RESPONSE W5 #31 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1004835
Saved       


[52/52] Submitting View postback for: ctl00$CPH$GridView2$ctl53$LinkButton1

--- VIEW RESPONSE W5 #52 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1365810
Saved       : gazette_output/w05_052_ctl00_CPH_GridView2_ctl53_LinkButton1_f84e69a18a.pdf

=== WINDOW 5 SUMMARY ===
Range         : 2023-08-01 to 2024-07-31
Total targets : 52
Downloaded    : 52
Failed/nonpdf : 0

========== WINDOW 6 ==========
From: 01/08/2024
To  : 31/07/2025

Submitting search request...

--- SEARCH RESPONSE WINDOW 6 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: text/html; charset=utf-8
Location    : None
Length      : 128970
Saved       : gazette_output/window_06_2024-08-01_to_2025-07-31_search_response.html

Hidden fields found in search response:
  ScriptManager1_HiddenField
  __EVENTARGUMENT
  __EVENTTARGET
  __LASTFOCUS
  __VIEWSTATE


[20/53] Submitting View postback for: ctl00$CPH$GridView2$ctl21$LinkButton1

--- VIEW RESPONSE W6 #20 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 928014
Saved       : gazette_output/w06_020_ctl00_CPH_GridView2_ctl21_LinkButton1_c09a262f67.pdf

[21/53] Submitting View postback for: ctl00$CPH$GridView2$ctl22$LinkButton1

--- VIEW RESPONSE W6 #21 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1376697
Saved       : gazette_output/w06_021_ctl00_CPH_GridView2_ctl22_LinkButton1_917bfdf476.pdf

[22/53] Submitting View postback for: ctl00$CPH$GridView2$ctl23$LinkButton1

--- VIEW RESPONSE W6 #22 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 555891
Saved       


[43/53] Submitting View postback for: ctl00$CPH$GridView2$ctl44$LinkButton1

--- VIEW RESPONSE W6 #43 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1329414
Saved       : gazette_output/w06_043_ctl00_CPH_GridView2_ctl44_LinkButton1_9849642d78.pdf

[44/53] Submitting View postback for: ctl00$CPH$GridView2$ctl45$LinkButton1

--- VIEW RESPONSE W6 #44 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1018645
Saved       : gazette_output/w06_044_ctl00_CPH_GridView2_ctl45_LinkButton1_9b0c7ba825.pdf

[45/53] Submitting View postback for: ctl00$CPH$GridView2$ctl46$LinkButton1

--- VIEW RESPONSE W6 #45 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1045841
Saved     


[10/21] Submitting View postback for: ctl00$CPH$GridView2$ctl11$LinkButton1

--- VIEW RESPONSE W7 #10 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 561605
Saved       : gazette_output/w07_010_ctl00_CPH_GridView2_ctl11_LinkButton1_6c0d20b321.pdf

[11/21] Submitting View postback for: ctl00$CPH$GridView2$ctl12$LinkButton1

--- VIEW RESPONSE W7 #11 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 1044410
Saved       : gazette_output/w07_011_ctl00_CPH_GridView2_ctl12_LinkButton1_5de5e17682.pdf

[12/21] Submitting View postback for: ctl00$CPH$GridView2$ctl13$LinkButton1

--- VIEW RESPONSE W7 #12 ---
URL         : https://egazzete.mahaonline.gov.in/Forms/GazetteSearch.aspx
Status      : 200
Content-Type: application/pdf
Location    : None
Length      : 741242
Saved       